In [1]:
!pip install llama-index-retrievers-bm25 llama-index-llms-gemini llama-index-postprocessor-sentence-transformers

ERROR: Ignored the following versions that require a different python version: 0.0.1 Requires-Python >=3.8.1,<3.12; 0.0.1 Requires-Python >=3.9,<3.12; 0.1.0 Requires-Python >=3.8.1,<3.12; 0.1.0 Requires-Python >=3.9,<3.12; 0.1.1 Requires-Python >=3.8.1,<3.12; 0.1.1 Requires-Python >=3.9,<3.12; 0.1.2 Requires-Python >=3.8.1,<3.12; 0.1.2 Requires-Python >=3.9,<3.12
ERROR: Could not find a version that satisfies the requirement llama-index-postprocessor-sentence-transformers (from versions: none)
ERROR: No matching distribution found for llama-index-postprocessor-sentence-transformers


In [2]:
!curl -sSL https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!./.venv/bin/python get-pip.py
!rm get-pip.py

zsh:1: no such file or directory: ./.venv/bin/python


In [1]:
%pip install llama-index llama-index-embeddings-ollama pinecone-client llama-index-vector-stores-pinecone python-dotenv llama-index-retrievers-bm25 llama-index-llms-gemini llama-index-postprocessor-sbert-rerank sentence-transformers

  Using cached llama_index-0.14.22-py3-none-any.whl.metadata (14 kB)
  Using cached llama_index_retrievers_bm25-0.7.1-py3-none-any.whl.metadata (447 bytes)
  Using cached llama_index_llms_gemini-0.6.2-py3-none-any.whl.metadata (690 bytes)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached scikit_learn-1.8.0-cp313-cp313-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached protobuf-7.34.1-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached cffi-2.0.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
  Using cac

In [21]:
%pip install llama-index-postprocessor-rankgpt-rerank
%pip install llama-index-vector-stores-pinecone pinecone-text


Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for mmh3: filename=mmh3-4.1.0-cp313-cp313-macosx_12_0_arm64.whl size=29244 sha256=878dbeea757e9dfd998d13a1375c08b3fee9ddea2e6fbb2cd10453c86b123ac4
  Stored in directory: /Users/raziqs/Library/Caches/pip/wheels/57/68/ca/1b7f541c188850a92dc3011ea464115ab2171bff64606c54f5
Successfully built mmh3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pinecone-text]
Note: you may need to restart the kernel to use updated packages.


In [13]:
%pip install llama-index-post-processor-gemini-rerank

ERROR: Could not find a version that satisfies the requirement llama-index-post-processor-gemini-rerank (from versions: none)
ERROR: No matching distribution found for llama-index-post-processor-gemini-rerank
Note: you may need to restart the kernel to use updated packages.


In [14]:
import os 
from dotenv import load_dotenv 
load_dotenv() 

from llama_index.llms.gemini import Gemini 
from llama_index.core.postprocessor import LLMRerank
from llama_index.core.query_engine import RetrieverQueryEngine 
from pinecone import Pinecone 
from llama_index.vector_stores.pinecone import PineconeVectorStore 
from llama_index.core import VectorStoreIndex

pinecone_api_key = os.getenv("PINECON_KEY") 
gemini_api_key = os.getenv("GEMINI_KEY") 

print("Initializing Pinecone and Gemini...") 

pc = Pinecone(api_key=pinecone_api_key) 

llm = Gemini( 
    api_key=gemini_api_key, 
    model="gemini-3.1-flash-lite", 
    temperature=0.0 
) 

# 2.point which index to use in pinecone, and set up the vector store wrapper for llama-index
pinecone_index = pc.Index("mykepatuhan") 
vector_store = PineconeVectorStore(
    pinecone_index=pinecone_index,
    add_sparse_vector=False  # set to true for hybrid search (must use dot-product similarity in pinecone)
) 

index = VectorStoreIndex.from_vector_store(vector_store=vector_store) 

print("Setting up Native Cloud Hybrid Retriever...") 

hybrid_retriever = index.as_retriever(
    vector_store_query_mode="default",  #set to "hybrid" for hybrid search 
    similarity_top_k=10
)

reranker = LLMRerank(
    llm=llm, 
    top_n=3 
)

print("Assembling the final Trust Engine...") 
query_engine = RetrieverQueryEngine.from_args( 
    retriever=hybrid_retriever, 
    node_postprocessors=[reranker], 
    llm=llm 
) 

# ========================================== 
print("\n" + "="*50) 
print("SYSTEM READY. RUNNING TEST QUERY...") 
print("="*50) 

test_question = "berikan 3 GARIS PANDUAN LESEN PERNIAGAAN DAN PERINDUSTRIAN?" 
print(f"User Question: {test_question}\n") 

# Run the pipeline! 
response = query_engine.query(test_question) 

print("--- GEMINI RESPONSE ---") 
print(response.response) 

print("\n--- EXACT CITATIONS USED ---") 
for i, source_node in enumerate(response.source_nodes): 
    meta = source_node.node.metadata 
    print(f"[{i+1}] Authority: {meta.get('authority', 'Unknown')} | Type: {meta.get('document_type', 'Unknown')} | Score: {source_node.score:.4f}")


Initializing Pinecone and Gemini...
Setting up Native Cloud Hybrid Retriever...
Assembling the final Trust Engine...

SYSTEM READY. RUNNING TEST QUERY...
User Question: berikan 3 GARIS PANDUAN LESEN PERNIAGAAN DAN PERINDUSTRIAN?



/var/folders/my/2fkdjjvj7h9dn3nmk1ttlccc0000gn/T/ipykernel_6762/1907509784.py:19: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  llm = Gemini(


--- GEMINI RESPONSE ---
Berikut adalah 3 garis panduan lesen perniagaan dan perindustrian:

1. Dilarang meletakkan sebarang peralatan atau barang-barang di kaki lima.
2. Menjaga kebersihan di dalam dan di luar bangunan atau premis.
3. Menyediakan kotak 'First-Aid' yang lengkap dan ditempatkan di bahagian yang bersesuaian.

--- EXACT CITATIONS USED ---
[1] Authority: MPKj | Type: guideline | Score: 10.0000
[2] Authority: MPKj | Type: guideline | Score: 10.0000
